# 204 — TCGA Confounder Assessment

## Objective

Characterize the principal biological and technical confounders associated
with the frozen TCGA multi-omic cohort generated in notebook 203.

The analysis will evaluate:

- tumor lineage;
- DNA-methylation platform;
- tumor purity;
- immune and stromal infiltration;
- proliferation;
- available technical and batch-associated variables.

This notebook does not repeat upstream RNA-seq, methylation, or multi-omic
quality control. It does not perform program discovery, definitive
residualization, or sample exclusion based solely on confounder values.

The outputs will provide the covariate framework required for lineage-aware
program discovery and robustness analyses in notebooks 205 and 206.

In [1]:
# =============================================================================
# Imports
# =============================================================================

from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

In [2]:
# =============================================================================
# Project root and paths
# =============================================================================

PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "pancancer-epigenetics":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

In [3]:
# =============================================================================
# Load final multi-omic consumables metadata
# =============================================================================

FINAL_CONSUMABLES_METADATA_PATH = (
    Paths.metadata
    / "tcga_primary_tumor_multiomic_final_consumables_metadata.json"
)

with FINAL_CONSUMABLES_METADATA_PATH.open("r", encoding="utf-8") as file:
    final_consumables_metadata = json.load(file)

final_artifact_paths = {
    name: PROJECT_ROOT / record["path"]
    for name, record in final_consumables_metadata["final_artifacts"].items()
}

In [4]:
# =============================================================================
# Load final case-level sample mapping
# =============================================================================

final_sample_mapping = pd.read_csv(
    final_artifact_paths["sample_mapping"],
    low_memory=False,
)

print(f"Final cases: {len(final_sample_mapping):,}")

display(
    pd.DataFrame(
        {
            "column": final_sample_mapping.columns,
            "dtype": final_sample_mapping.dtypes.astype(str).values,
        }
    )
)

Final cases: 9,965


,column,dtype
0,final_sample_column_index,int64
1,case_submitter_id,str
2,sample_submitter_id,str
3,project_id,str
4,rna_case_uuid,str
5,rna_aliquot_uuid,str
6,rna_aliquot_submitter_id,str
7,rna_file_id,str
8,rna_file_name,str
9,rna_candidate_matrix_column_index,int64


In [5]:
# =============================================================================
# Build base case-level confounder table
# =============================================================================

confounder_table = final_sample_mapping[
    [
        "final_sample_column_index",
        "case_submitter_id",
        "sample_submitter_id",
        "project_id",
        "methylation_platform",
        "gene_assigned_fraction_of_accounted",
        "missing_beta_fraction",
    ]
].copy()

confounder_table.head()

,final_sample_column_index,case_submitter_id,sample_submitter_id,project_id,methylation_platform,gene_assigned_fraction_of_accounted,missing_beta_fraction
0,0,TCGA-02-0003,TCGA-02-0003-01A,TCGA-GBM,Illumina Human Methylation 27,0.439303,0.175792
1,1,TCGA-02-0033,TCGA-02-0033-01A,TCGA-GBM,Illumina Human Methylation 27,0.513737,0.099463
2,2,TCGA-02-0038,TCGA-02-0038-01A,TCGA-GBM,Illumina Human Methylation 27,0.325748,0.113387
3,3,TCGA-02-0047,TCGA-02-0047-01A,TCGA-GBM,Illumina Human Methylation 27,0.669034,0.140039
4,4,TCGA-02-0055,TCGA-02-0055-01A,TCGA-GBM,Illumina Human Methylation 27,0.722150,0.110740


In [6]:
# =============================================================================
# Summarize cohort composition
# =============================================================================

project_counts = (
    confounder_table["project_id"]
    .value_counts()
    .rename_axis("project_id")
    .reset_index(name="n_cases")
)

platform_counts = (
    confounder_table["methylation_platform"]
    .value_counts()
    .rename_axis("methylation_platform")
    .reset_index(name="n_cases")
)

display(project_counts)
display(platform_counts)

,project_id,n_cases
0,TCGA-BRCA,1089
1,TCGA-UCEC,543
2,TCGA-HNSC,520
3,TCGA-LGG,513
4,TCGA-LUAD,512
5,TCGA-THCA,505
6,TCGA-LUSC,500
7,TCGA-PRAD,496
8,TCGA-KIRC,487
9,TCGA-COAD,453


,methylation_platform,n_cases
0,Illumina Human Methylation 450,8345
1,Illumina Human Methylation 27,1620


In [7]:
# =============================================================================
# Summarize methylation platform by project
# =============================================================================

platform_by_project = pd.crosstab(
    confounder_table["project_id"],
    confounder_table["methylation_platform"],
)

platform_fraction_by_project = platform_by_project.div(
    platform_by_project.sum(axis=1),
    axis=0,
)

display(platform_by_project)
display(platform_fraction_by_project.round(3))

methylation_platform,Illumina Human Methylation 27,Illumina Human Methylation 450
project_id,,
TCGA-ACC,0,79
TCGA-BLCA,0,405
TCGA-BRCA,312,777
TCGA-CESC,0,304
TCGA-CHOL,0,35
TCGA-COAD,160,293
TCGA-DLBC,0,48
TCGA-ESCA,0,184
TCGA-GBM,150,80


methylation_platform,Illumina Human Methylation 27,Illumina Human Methylation 450
project_id,,
TCGA-ACC,0.000,1.000
TCGA-BLCA,0.000,1.000
TCGA-BRCA,0.287,0.713
TCGA-CESC,0.000,1.000
TCGA-CHOL,0.000,1.000
TCGA-COAD,0.353,0.647
TCGA-DLBC,0.000,1.000
TCGA-ESCA,0.000,1.000
TCGA-GBM,0.652,0.348


In [8]:
# =============================================================================
# Classify project-level methylation platform composition
# =============================================================================

project_platform_summary = platform_by_project.assign(
    n_cases=platform_by_project.sum(axis=1),
    mixed_platform=platform_by_project.gt(0).sum(axis=1).gt(1),
    dominant_platform=platform_by_project.idxmax(axis=1),
).reset_index()

display(project_platform_summary)

methylation_platform,project_id,Illumina Human Methylation 27,Illumina Human Methylation 450,n_cases,mixed_platform,dominant_platform
0,TCGA-ACC,0,79,79,False,Illumina Human Methylation 450
1,TCGA-BLCA,0,405,405,False,Illumina Human Methylation 450
2,TCGA-BRCA,312,777,1089,True,Illumina Human Methylation 450
3,TCGA-CESC,0,304,304,False,Illumina Human Methylation 450
4,TCGA-CHOL,0,35,35,False,Illumina Human Methylation 450
5,TCGA-COAD,160,293,453,True,Illumina Human Methylation 450
6,TCGA-DLBC,0,48,48,False,Illumina Human Methylation 450
7,TCGA-ESCA,0,184,184,False,Illumina Human Methylation 450
8,TCGA-GBM,150,80,230,True,Illumina Human Methylation 27
9,TCGA-HNSC,0,520,520,False,Illumina Human Methylation 450


In [9]:
# =============================================================================
# Summarize technical covariates by project and platform
# =============================================================================

technical_summary = (
    confounder_table
    .groupby(
        ["project_id", "methylation_platform"],
        observed=True,
    )
    .agg(
        n_cases=("case_submitter_id", "size"),
        median_rna_assigned_fraction=(
            "gene_assigned_fraction_of_accounted",
            "median",
        ),
        median_missing_beta_fraction=(
            "missing_beta_fraction",
            "median",
        ),
    )
    .reset_index()
)

display(technical_summary)

,project_id,methylation_platform,n_cases,median_rna_assigned_fraction,median_missing_beta_fraction
0,TCGA-ACC,Illumina Human Methylation 450,79,0.750312,0.149093
1,TCGA-BLCA,Illumina Human Methylation 450,405,0.761461,0.150639
2,TCGA-BRCA,Illumina Human Methylation 27,312,0.774572,0.101494
3,TCGA-BRCA,Illumina Human Methylation 450,777,0.773205,0.149531
4,TCGA-CESC,Illumina Human Methylation 450,304,0.764001,0.147737
5,TCGA-CHOL,Illumina Human Methylation 450,35,0.762584,0.153949
6,TCGA-COAD,Illumina Human Methylation 27,160,0.757137,0.103615
7,TCGA-COAD,Illumina Human Methylation 450,293,0.763942,0.142887
8,TCGA-DLBC,Illumina Human Methylation 450,48,0.725527,0.153274
9,TCGA-ESCA,Illumina Human Methylation 450,184,0.644783,0.147864


## External confounder resources

External TCGA confounder resources were manually downloaded and preserved in:

`data/raw/tcga/confounders/`

The available resources include:

- ESTIMATE stromal, immune, and purity-related scores;
- consensus tumor-purity estimates;
- PanCanAtlas leukocyte-fraction estimates;
- ABSOLUTE purity and genomic-composition estimates.

Complete provenance, publication references, DOI identifiers, source URLs,
file roles, and canonical local paths are documented in:

`config/raw_data_registry.json`

These resources are treated as candidate covariate sources. Their sample-level
coverage and identifier compatibility with the frozen 9,965-case multi-omic
cohort will be assessed before selecting variables for downstream analyses.

In [10]:
# =============================================================================
# External confounder source paths
# =============================================================================

CONFOUNDER_RAW_DIR = Paths.tcga / "confounders"

ESTIMATE_SCORES_PATH = (
    CONFOUNDER_RAW_DIR
    / "41467_2013_BFncomms3612_MOESM489_ESM.xlsx"
)

CONSENSUS_PURITY_PATH = (
    CONFOUNDER_RAW_DIR
    / "41467_2015_BFncomms9971_MOESM1236_ESM.xlsx"
)

LEUKOCYTE_FRACTION_PATH = (
    CONFOUNDER_RAW_DIR
    / "TCGA_all_leuk_estimate.masked.20170107.tsv"
)

ABSOLUTE_ESTIMATES_PATH = (
    CONFOUNDER_RAW_DIR
    / "TCGA_mastercalls.abs_tables_JSedit.fixed.txt"
)

In [11]:
# =============================================================================
# Load consensus tumor-purity estimates
# =============================================================================

consensus_purity = pd.read_excel(
    CONSENSUS_PURITY_PATH,
    sheet_name="Supp Data 1",
    header=3,
    usecols="A:G",
).rename(
    columns={
        "Sample ID": "sample_submitter_id",
        "Cancer type": "cancer_type",
        "ESTIMATE": "estimate_purity",
        "ABSOLUTE": "absolute_purity_publication",
        "LUMP": "lump_purity",
        "IHC": "ihc_purity",
        "CPE": "consensus_purity_estimate",
    }
)

print(f"Consensus-purity records: {len(consensus_purity):,}")
display(consensus_purity.head())

c:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Consensus-purity records: 9,364


,sample_submitter_id,cancer_type,estimate_purity,absolute_purity_publication,lump_purity,ihc_purity,consensus_purity_estimate
0,TCGA-OR-A5J1-01A,ACC,0.9368,NaN,0.9774,0.80,0.9246
1,TCGA-OR-A5J2-01A,ACC,0.9175,NaN,0.6174,0.95,0.8985
2,TCGA-OR-A5J3-01A,ACC,0.9670,NaN,0.9249,0.80,0.9466
3,TCGA-OR-A5J4-01A,ACC,NaN,NaN,0.9199,0.80,0.8660
4,TCGA-OR-A5J5-01A,ACC,0.9761,NaN,1.0000,0.80,0.9780


In [12]:
# =============================================================================
# Assess consensus-purity coverage in the frozen cohort
# =============================================================================

cpe_sample_ids = set(consensus_purity["sample_submitter_id"])

cpe_coverage = confounder_table["sample_submitter_id"].isin(
    cpe_sample_ids
)

print(
    f"Unique source samples: "
    f"{consensus_purity['sample_submitter_id'].nunique():,}"
)
print(f"Frozen cases with CPE record: {cpe_coverage.sum():,}")
print(f"Frozen cases without CPE record: {(~cpe_coverage).sum():,}")
print(f"Coverage: {cpe_coverage.mean():.2%}")

Unique source samples: 9,364
Frozen cases with CPE record: 8,099
Frozen cases without CPE record: 1,866
Coverage: 81.27%


In [13]:
# =============================================================================
# Distinguish exact-sample and case-level CPE availability
# =============================================================================

cpe_case_ids = set(
    consensus_purity["sample_submitter_id"].str[:12]
)

same_case_available = (
    confounder_table["case_submitter_id"].isin(cpe_case_ids)
)

print(f"Exact sample matches: {cpe_coverage.sum():,}")
print(
    "Same case, different sample only: "
    f"{(same_case_available & ~cpe_coverage).sum():,}"
)
print(
    "Case absent from CPE resource: "
    f"{(~same_case_available).sum():,}"
)

Exact sample matches: 8,099
Same case, different sample only: 0
Case absent from CPE resource: 1,866


In [14]:
# =============================================================================
# Add consensus-purity estimates to the confounder table
# =============================================================================

purity_columns = [
    "sample_submitter_id",
    "estimate_purity",
    "absolute_purity_publication",
    "lump_purity",
    "ihc_purity",
    "consensus_purity_estimate",
]

confounder_table = confounder_table.merge(
    consensus_purity[purity_columns],
    on="sample_submitter_id",
    how="left",
)

print(
    "Cases with non-missing CPE: "
    f"{confounder_table['consensus_purity_estimate'].notna().sum():,}"
)
print(
    "Cases without CPE: "
    f"{confounder_table['consensus_purity_estimate'].isna().sum():,}"
)

Cases with non-missing CPE: 8,060
Cases without CPE: 1,905


In [15]:
# =============================================================================
# Load PanCanAtlas ABSOLUTE estimates
# =============================================================================

absolute_estimates = pd.read_csv(
    ABSOLUTE_ESTIMATES_PATH,
    sep="\t",
)

absolute_estimates["sample_submitter_id"] = (
    absolute_estimates["sample"].str[:16]
)

absolute_estimates = absolute_estimates.rename(
    columns={
        "call status": "absolute_call_status",
        "purity": "absolute_purity",
        "ploidy": "absolute_ploidy",
        "Genome doublings": "genome_doublings",
        "Cancer DNA fraction": "cancer_dna_fraction",
        "Subclonal genome fraction": "subclonal_genome_fraction",
    }
)

print(f"ABSOLUTE records: {len(absolute_estimates):,}")
display(
    absolute_estimates[
        [
            "sample_submitter_id",
            "absolute_call_status",
            "absolute_purity",
            "absolute_ploidy",
        ]
    ].head()
)

ABSOLUTE records: 10,786


,sample_submitter_id,absolute_call_status,absolute_purity,absolute_ploidy
0,TCGA-OR-A5J1-01A,called,0.90,2.00
1,TCGA-OR-A5J2-01A,called,0.89,1.30
2,TCGA-OR-A5J3-01A,called,0.93,1.27
3,TCGA-OR-A5J4-01A,called,0.87,2.60
4,TCGA-OR-A5J5-01A,called,0.93,2.79


In [16]:
# =============================================================================
# Assess ABSOLUTE sample multiplicity and cohort coverage
# =============================================================================

absolute_sample_counts = (
    absolute_estimates["sample_submitter_id"].value_counts()
)

absolute_coverage = confounder_table["sample_submitter_id"].isin(
    absolute_sample_counts.index
)

print(f"Unique ABSOLUTE samples: {len(absolute_sample_counts):,}")
print(
    "Samples with multiple ABSOLUTE records: "
    f"{absolute_sample_counts.gt(1).sum():,}"
)
print(f"Maximum records per sample: {absolute_sample_counts.max():,}")
print(f"Frozen cases with ABSOLUTE record: {absolute_coverage.sum():,}")
print(f"Frozen cases without ABSOLUTE record: {(~absolute_coverage).sum():,}")
print(f"Coverage: {absolute_coverage.mean():.2%}")

Unique ABSOLUTE samples: 10,600
Samples with multiple ABSOLUTE records: 85
Maximum records per sample: 11
Frozen cases with ABSOLUTE record: 9,080
Frozen cases without ABSOLUTE record: 885
Coverage: 91.12%


In [17]:
# =============================================================================
# Assess ABSOLUTE ambiguity in the frozen cohort
# =============================================================================

duplicate_absolute_samples = set(
    absolute_sample_counts[
        absolute_sample_counts.gt(1)
    ].index
)

absolute_aliquot_ids = set(absolute_estimates["sample"])

duplicate_prefix_match = (
    confounder_table["sample_submitter_id"]
    .isin(duplicate_absolute_samples)
)

exact_methylation_aliquot_match = (
    final_sample_mapping["methylation_aliquot_submitter_id"]
    .isin(absolute_aliquot_ids)
)

exact_rna_aliquot_match = (
    final_sample_mapping["rna_aliquot_submitter_id"]
    .isin(absolute_aliquot_ids)
)

print(
    "Frozen cases matching duplicated ABSOLUTE samples: "
    f"{duplicate_prefix_match.sum():,}"
)
print(
    "Exact methylation-aliquot matches: "
    f"{exact_methylation_aliquot_match.sum():,}"
)
print(
    "Exact RNA-aliquot matches: "
    f"{exact_rna_aliquot_match.sum():,}"
)

Frozen cases matching duplicated ABSOLUTE samples: 0
Exact methylation-aliquot matches: 0
Exact RNA-aliquot matches: 0


In [18]:
# =============================================================================
# Rebuild ABSOLUTE matching with the TCGA sample-level key
# =============================================================================

absolute_estimates = absolute_estimates.drop(
    columns="sample_submitter_id",
    errors="ignore",
)

absolute_estimates["tcga_sample_key"] = (
    absolute_estimates["array"]
    .astype("string")
    .str.strip()
)

frozen_sample_keys = (
    confounder_table["sample_submitter_id"]
    .str[:15]
)

absolute_key_counts = (
    absolute_estimates["tcga_sample_key"]
    .value_counts()
)

absolute_coverage = frozen_sample_keys.isin(
    absolute_key_counts.index
)

matched_duplicate_keys = frozen_sample_keys.isin(
    absolute_key_counts[
        absolute_key_counts.gt(1)
    ].index
)

print(f"Unique ABSOLUTE sample keys: {len(absolute_key_counts):,}")
print(
    "Duplicated ABSOLUTE sample keys: "
    f"{absolute_key_counts.gt(1).sum():,}"
)
print(
    "Frozen cases with ABSOLUTE record: "
    f"{absolute_coverage.sum():,}"
)
print(
    "Frozen cases matching duplicated keys: "
    f"{matched_duplicate_keys.sum():,}"
)
print(f"Coverage: {absolute_coverage.mean():.2%}")

Unique ABSOLUTE sample keys: 10,786
Duplicated ABSOLUTE sample keys: 0
Frozen cases with ABSOLUTE record: 9,649
Frozen cases matching duplicated keys: 0
Coverage: 96.83%


In [19]:
# =============================================================================
# Summarize ABSOLUTE call status in the frozen cohort
# =============================================================================

absolute_in_cohort = absolute_estimates[
    absolute_estimates["tcga_sample_key"].isin(frozen_sample_keys)
].copy()

absolute_status_summary = (
    absolute_in_cohort
    .groupby("absolute_call_status", dropna=False)
    .agg(
        n_cases=("tcga_sample_key", "size"),
        n_with_purity=(
            "absolute_purity",
            lambda values: values.notna().sum(),
        ),
    )
    .reset_index()
)

absolute_status_summary["purity_coverage"] = (
    absolute_status_summary["n_with_purity"]
    / absolute_status_summary["n_cases"]
)

display(absolute_status_summary)

,absolute_call_status,n_cases,n_with_purity,purity_coverage
0,called,8955,8955,1.0
1,legacy_call,362,362,1.0
2,legacy_maf_call,33,33,1.0
3,maf_call,135,135,1.0
4,snp_call,36,36,1.0
5,NaN,128,0,0.0


In [20]:
# =============================================================================
# Add ABSOLUTE purity estimates to the confounder table
# =============================================================================

absolute_for_merge = absolute_estimates[
    [
        "tcga_sample_key",
        "absolute_call_status",
        "absolute_purity",
    ]
].copy()

confounder_table["tcga_sample_key"] = (
    confounder_table["sample_submitter_id"].str[:15]
)

confounder_table = confounder_table.merge(
    absolute_for_merge,
    on="tcga_sample_key",
    how="left",
)

print(
    "Cases with ABSOLUTE purity: "
    f"{confounder_table['absolute_purity'].notna().sum():,}"
)
print(
    "Cases with strict called status: "
    f"{confounder_table['absolute_call_status'].eq('called').sum():,}"
)
print(
    "Cases without ABSOLUTE purity: "
    f"{confounder_table['absolute_purity'].isna().sum():,}"
)

Cases with ABSOLUTE purity: 9,521
Cases with strict called status: 8,955
Cases without ABSOLUTE purity: 444


In [21]:
# =============================================================================
# Summarize joint tumor-purity coverage
# =============================================================================

has_cpe = confounder_table[
    "consensus_purity_estimate"
].notna()

has_absolute = confounder_table[
    "absolute_purity"
].notna()

confounder_table["purity_availability"] = np.select(
    [
        has_cpe & has_absolute,
        has_cpe & ~has_absolute,
        ~has_cpe & has_absolute,
    ],
    [
        "CPE and ABSOLUTE",
        "CPE only",
        "ABSOLUTE only",
    ],
    default="Neither",
)

purity_coverage_summary = (
    confounder_table["purity_availability"]
    .value_counts()
    .rename_axis("purity_availability")
    .reset_index(name="n_cases")
)

purity_coverage_summary["fraction"] = (
    purity_coverage_summary["n_cases"]
    / len(confounder_table)
)

display(purity_coverage_summary)

,purity_availability,n_cases,fraction
0,CPE and ABSOLUTE,7781,0.780833
1,ABSOLUTE only,1740,0.174611
2,CPE only,279,0.027998
3,Neither,165,0.016558


In [22]:
# =============================================================================
# Summarize tumor-purity coverage by TCGA project
# =============================================================================

purity_coverage_by_project = (
    confounder_table
    .groupby("project_id", observed=True)
    .agg(
        n_cases=("case_submitter_id", "size"),
        n_absolute=("absolute_purity", "count"),
        n_cpe=("consensus_purity_estimate", "count"),
        n_without_either=(
            "purity_availability",
            lambda values: values.eq("Neither").sum(),
        ),
    )
    .reset_index()
)

purity_coverage_by_project["absolute_fraction"] = (
    purity_coverage_by_project["n_absolute"]
    / purity_coverage_by_project["n_cases"]
)

purity_coverage_by_project["cpe_fraction"] = (
    purity_coverage_by_project["n_cpe"]
    / purity_coverage_by_project["n_cases"]
)

display(
    purity_coverage_by_project.sort_values(
        "absolute_fraction"
    ).reset_index(drop=True)
)

,project_id,n_cases,n_absolute,n_cpe,n_without_either,absolute_fraction,cpe_fraction
0,TCGA-LAML,134,90,0,44,0.671642,0.000000
1,TCGA-THYM,120,103,0,17,0.858333,0.000000
2,TCGA-ESCA,184,161,0,23,0.875000,0.000000
3,TCGA-PAAD,178,158,0,20,0.887640,0.000000
4,TCGA-PCPG,179,161,0,18,0.899441,0.000000
5,TCGA-THCA,505,463,493,4,0.916832,0.976238
6,TCGA-MESO,87,81,0,6,0.931034,0.000000
7,TCGA-KIRC,487,455,486,0,0.934292,0.997947
8,TCGA-SARC,259,242,0,17,0.934363,0.000000
9,TCGA-PRAD,496,471,495,0,0.949597,0.997984


In [23]:
# =============================================================================
# Load PanImmune leukocyte-fraction estimates
# =============================================================================

leukocyte_fraction = pd.read_csv(
    LEUKOCYTE_FRACTION_PATH,
    sep="\t",
    header=None,
    names=[
        "cancer_type",
        "aliquot_submitter_id",
        "leukocyte_fraction",
    ],
)

leukocyte_fraction["sample_submitter_id"] = (
    leukocyte_fraction["aliquot_submitter_id"]
    .astype("string")
    .str.strip()
    .str[:16]
)

print(f"PanImmune records: {len(leukocyte_fraction):,}")

display(
    leukocyte_fraction[
        [
            "cancer_type",
            "sample_submitter_id",
            "aliquot_submitter_id",
            "leukocyte_fraction",
        ]
    ].head()
)

PanImmune records: 10,817


,cancer_type,sample_submitter_id,aliquot_submitter_id,leukocyte_fraction
0,ACC,TCGA-OR-A5J1-01A,TCGA-OR-A5J1-01A-11D-A29J-05,0.046374
1,ACC,TCGA-OR-A5J2-01A,TCGA-OR-A5J2-01A-11D-A29J-05,0.057859
2,ACC,TCGA-OR-A5J3-01A,TCGA-OR-A5J3-01A-11D-A29J-05,0.048460
3,ACC,TCGA-OR-A5J4-01A,TCGA-OR-A5J4-01A-11D-A29J-05,0.043988
4,ACC,TCGA-OR-A5J5-01A,TCGA-OR-A5J5-01A-11D-A29J-05,0.016759


In [24]:
print(f"PanImmune records: {len(leukocyte_fraction):,}")

print(
    leukocyte_fraction[
        [
            "cancer_type",
            "sample_submitter_id",
            "aliquot_submitter_id",
            "leukocyte_fraction",
        ]
    ].head()
)

PanImmune records: 10,817
  cancer_type sample_submitter_id          aliquot_submitter_id  \
0         ACC    TCGA-OR-A5J1-01A  TCGA-OR-A5J1-01A-11D-A29J-05   
1         ACC    TCGA-OR-A5J2-01A  TCGA-OR-A5J2-01A-11D-A29J-05   
2         ACC    TCGA-OR-A5J3-01A  TCGA-OR-A5J3-01A-11D-A29J-05   
3         ACC    TCGA-OR-A5J4-01A  TCGA-OR-A5J4-01A-11D-A29J-05   
4         ACC    TCGA-OR-A5J5-01A  TCGA-OR-A5J5-01A-11D-A29J-05   

   leukocyte_fraction  
0            0.046374  
1            0.057859  
2            0.048460  
3            0.043988  
4            0.016759  


In [25]:
# =============================================================================
# Assess PanImmune sample multiplicity and cohort coverage
# =============================================================================

leukocyte_sample_counts = (
    leukocyte_fraction["sample_submitter_id"]
    .value_counts()
)

leukocyte_coverage = (
    confounder_table["sample_submitter_id"]
    .isin(leukocyte_sample_counts.index)
)

print(
    "Unique PanImmune samples: "
    f"{len(leukocyte_sample_counts):,}"
)
print(
    "Samples with multiple PanImmune records: "
    f"{leukocyte_sample_counts.gt(1).sum():,}"
)
print(
    "Maximum records per sample: "
    f"{leukocyte_sample_counts.max():,}"
)
print(
    "Frozen cases with leukocyte fraction: "
    f"{leukocyte_coverage.sum():,}"
)
print(
    "Frozen cases without leukocyte fraction: "
    f"{(~leukocyte_coverage).sum():,}"
)
print(f"Coverage: {leukocyte_coverage.mean():.2%}")

Unique PanImmune samples: 10,770
Samples with multiple PanImmune records: 42
Maximum records per sample: 6
Frozen cases with leukocyte fraction: 9,631
Frozen cases without leukocyte fraction: 334
Coverage: 96.65%


In [26]:
# =============================================================================
# Characterize duplicated PanImmune sample records
# =============================================================================

duplicate_leukocyte_samples = set(
    leukocyte_sample_counts[
        leukocyte_sample_counts.gt(1)
    ].index
)

frozen_duplicate_leukocyte_samples = set(
    confounder_table.loc[
        confounder_table["sample_submitter_id"].isin(
            duplicate_leukocyte_samples
        ),
        "sample_submitter_id",
    ]
)

leukocyte_duplicate_summary = (
    leukocyte_fraction[
        leukocyte_fraction["sample_submitter_id"].isin(
            frozen_duplicate_leukocyte_samples
        )
    ]
    .groupby("sample_submitter_id")
    .agg(
        n_records=("leukocyte_fraction", "size"),
        n_unique_values=("leukocyte_fraction", "nunique"),
        minimum=("leukocyte_fraction", "min"),
        maximum=("leukocyte_fraction", "max"),
    )
    .reset_index()
)

leukocyte_duplicate_summary["value_range"] = (
    leukocyte_duplicate_summary["maximum"]
    - leukocyte_duplicate_summary["minimum"]
)

print(
    "Frozen samples affected by duplicated records: "
    f"{len(frozen_duplicate_leukocyte_samples):,}"
)

display(
    leukocyte_duplicate_summary.sort_values(
        "value_range",
        ascending=False,
    )
)

Frozen samples affected by duplicated records: 32


,sample_submitter_id,n_records,n_unique_values,minimum,maximum,value_range
22,TCGA-AG-A026-01A,2,2,0.037361,0.166078,0.128716
14,TCGA-A6-3809-01A,2,2,0.269124,0.329221,0.060096
25,TCGA-B2-3924-01A,2,2,0.304483,0.351870,0.047387
24,TCGA-AK-3453-01A,2,2,0.015664,0.061336,0.045672
7,TCGA-44-4112-01A,2,2,0.213551,0.254616,0.041065
0,TCGA-06-0125-01A,2,2,0.052606,0.085874,0.033268
21,TCGA-A7-A26E-01A,2,2,0.121300,0.152006,0.030706
3,TCGA-21-1076-01A,2,2,0.473299,0.503455,0.030156
12,TCGA-A6-2677-01A,2,2,0.027898,0.057167,0.029269
29,TCGA-BK-A26L-01A,2,2,0.089873,0.118654,0.028781


In [27]:
# =============================================================================
# Resolve duplicated PanImmune records by methylation aliquot
# =============================================================================

panimmune_duplicate_candidates = (
    final_sample_mapping[
        [
            "sample_submitter_id",
            "methylation_aliquot_submitter_id",
        ]
    ]
    .merge(
        leukocyte_fraction[
            [
                "sample_submitter_id",
                "aliquot_submitter_id",
                "leukocyte_fraction",
            ]
        ],
        on="sample_submitter_id",
        how="inner",
    )
)

panimmune_duplicate_candidates = (
    panimmune_duplicate_candidates[
        panimmune_duplicate_candidates["sample_submitter_id"].isin(
            frozen_duplicate_leukocyte_samples
        )
    ]
    .copy()
)

panimmune_duplicate_candidates["exact_aliquot_match"] = (
    panimmune_duplicate_candidates["aliquot_submitter_id"]
    == panimmune_duplicate_candidates[
        "methylation_aliquot_submitter_id"
    ]
)

panimmune_duplicate_resolution = (
    panimmune_duplicate_candidates
    .groupby("sample_submitter_id")
    .agg(
        n_records=("aliquot_submitter_id", "size"),
        n_exact_matches=("exact_aliquot_match", "sum"),
    )
    .reset_index()
)

display(
    panimmune_duplicate_resolution[
        "n_exact_matches"
    ].value_counts().sort_index()
)

display(
    panimmune_duplicate_resolution[
        panimmune_duplicate_resolution["n_exact_matches"].ne(1)
    ]
)

n_exact_matches
1    32
Name: count, dtype: int64

,sample_submitter_id,n_records,n_exact_matches


In [28]:
# =============================================================================
# Add PanImmune leukocyte fraction to the confounder table
# =============================================================================

panimmune_unique = leukocyte_fraction[
    leukocyte_fraction["sample_submitter_id"].map(
        leukocyte_sample_counts
    ).eq(1)
][
    ["sample_submitter_id", "leukocyte_fraction"]
]

panimmune_resolved_duplicates = (
    panimmune_duplicate_candidates[
        panimmune_duplicate_candidates["exact_aliquot_match"]
    ][
        ["sample_submitter_id", "leukocyte_fraction"]
    ]
)

panimmune_for_merge = pd.concat(
    [
        panimmune_unique,
        panimmune_resolved_duplicates,
    ],
    ignore_index=True,
)

confounder_table = confounder_table.merge(
    panimmune_for_merge,
    on="sample_submitter_id",
    how="left",
)

print(
    "Cases with leukocyte fraction: "
    f"{confounder_table['leukocyte_fraction'].notna().sum():,}"
)
print(
    "Cases without leukocyte fraction: "
    f"{confounder_table['leukocyte_fraction'].isna().sum():,}"
)

Cases with leukocyte fraction: 9,631
Cases without leukocyte fraction: 334


In [29]:
# =============================================================================
# Inspect ESTIMATE workbook
# =============================================================================

estimate_workbook = pd.ExcelFile(
    ESTIMATE_SCORES_PATH
)

print(estimate_workbook.sheet_names)

['Affymetrix', 'Agilent', 'RNASeq', 'RNASeqV2']


In [30]:
# =============================================================================
# Inspect ESTIMATE RNASeqV2 sheet
# =============================================================================

estimate_rnaseqv2_preview = pd.read_excel(
    ESTIMATE_SCORES_PATH,
    sheet_name="RNASeqV2",
    header=None,
    nrows=10,
)

display(estimate_rnaseqv2_preview)

,0,1,2,3,4,5,6
0,"Supplementary Data 2: A list of stromal, immun...",NaN,NaN,NaN,NaN,NaN,NaN
1,ID,Disease,Platform,Stromal score,Immune score,ESTIMATE score,tumor.purity
2,TCGA.04.1348.01,RNAseqV2,ovarian serous cystadenocarcinoma,-793.695034,858.26698,64.571946,0.76
3,TCGA.04.1357.01,RNAseqV2,ovarian serous cystadenocarcinoma,-70.305823,1794.027478,1723.721655,0.52
4,TCGA.04.1362.01,RNAseqV2,ovarian serous cystadenocarcinoma,-1255.427529,-403.746461,-1659.173989,0.87
5,TCGA.04.1364.01,RNAseqV2,ovarian serous cystadenocarcinoma,-1855.444916,-1790.631602,-3646.076518,0.91
6,TCGA.04.1365.01,RNAseqV2,ovarian serous cystadenocarcinoma,-951.973407,512.84288,-439.130527,0.82
7,TCGA.04.1514.01,RNAseqV2,ovarian serous cystadenocarcinoma,-1216.780113,-2073.578768,-3290.358881,0.92
8,TCGA.04.1519.01,RNAseqV2,ovarian serous cystadenocarcinoma,-1398.28559,-1406.627385,-2804.912975,0.99
9,TCGA.09.0364.01,RNAseqV2,ovarian serous cystadenocarcinoma,-1437.841201,-2209.521038,-3647.362239,0.89


In [31]:
# =============================================================================
# Load ESTIMATE RNASeqV2 scores
# =============================================================================

estimate_scores = pd.read_excel(
    ESTIMATE_SCORES_PATH,
    sheet_name="RNASeqV2",
    header=1,
).rename(
    columns={
        "ID": "estimate_id",
        "Disease": "expression_assay",
        "Platform": "cancer_type_label",
        "Stromal score": "stromal_score",
        "Immune score": "estimate_immune_score",
        "ESTIMATE score": "estimate_score",
        "tumor.purity": "estimate_tumor_purity",
    }
)

estimate_scores["tcga_sample_key"] = (
    estimate_scores["estimate_id"]
    .astype("string")
    .str.strip()
    .str.replace(".", "-", regex=False)
)

print(f"ESTIMATE RNASeqV2 records: {len(estimate_scores):,}")

display(
    estimate_scores[
        [
            "tcga_sample_key",
            "stromal_score",
            "estimate_immune_score",
            "estimate_score",
            "estimate_tumor_purity",
        ]
    ].head()
)

ESTIMATE RNASeqV2 records: 2,463


c:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,tcga_sample_key,stromal_score,estimate_immune_score,estimate_score,estimate_tumor_purity
0,TCGA-04-1348-01,-793.695034,858.266980,64.571946,0.76
1,TCGA-04-1357-01,-70.305823,1794.027478,1723.721655,0.52
2,TCGA-04-1362-01,-1255.427529,-403.746461,-1659.173989,0.87
3,TCGA-04-1364-01,-1855.444916,-1790.631602,-3646.076518,0.91
4,TCGA-04-1365-01,-951.973407,512.842880,-439.130527,0.82


In [32]:
# =============================================================================
# Assess ESTIMATE sample multiplicity and cohort coverage
# =============================================================================

estimate_key_counts = (
    estimate_scores["tcga_sample_key"]
    .value_counts()
)

estimate_coverage = (
    confounder_table["tcga_sample_key"]
    .isin(estimate_key_counts.index)
)

print(f"Unique ESTIMATE sample keys: {len(estimate_key_counts):,}")
print(
    "Duplicated ESTIMATE sample keys: "
    f"{estimate_key_counts.gt(1).sum():,}"
)
print(
    "Frozen cases with ESTIMATE scores: "
    f"{estimate_coverage.sum():,}"
)
print(
    "Frozen cases without ESTIMATE scores: "
    f"{(~estimate_coverage).sum():,}"
)
print(f"Coverage: {estimate_coverage.mean():.2%}")

Unique ESTIMATE sample keys: 2,463
Duplicated ESTIMATE sample keys: 0
Frozen cases with ESTIMATE scores: 2,399
Frozen cases without ESTIMATE scores: 7,566
Coverage: 24.07%


In [33]:
# =============================================================================
# Summarize ESTIMATE coverage by TCGA project
# =============================================================================

estimate_coverage_by_project = (
    confounder_table
    .assign(
        has_estimate=confounder_table["tcga_sample_key"].isin(
            estimate_key_counts.index
        )
    )
    .groupby("project_id", observed=True)
    .agg(
        n_cases=("case_submitter_id", "size"),
        n_estimate=("has_estimate", "sum"),
    )
    .reset_index()
)

estimate_coverage_by_project["coverage"] = (
    estimate_coverage_by_project["n_estimate"]
    / estimate_coverage_by_project["n_cases"]
)

display(
    estimate_coverage_by_project.sort_values(
        "coverage"
    ).reset_index(drop=True)
)

,project_id,n_cases,n_estimate,coverage
0,TCGA-ACC,79,0,0.000000
1,TCGA-CESC,304,0,0.000000
2,TCGA-DLBC,48,0,0.000000
3,TCGA-CHOL,35,0,0.000000
4,TCGA-ESCA,184,0,0.000000
5,TCGA-LAML,134,0,0.000000
6,TCGA-LGG,513,0,0.000000
7,TCGA-KICH,66,0,0.000000
8,TCGA-LIHC,371,0,0.000000
9,TCGA-KIRP,280,0,0.000000


In [34]:
# =============================================================================
# Summarize PanImmune coverage by TCGA project
# =============================================================================

leukocyte_coverage_by_project = (
    confounder_table
    .groupby("project_id", observed=True)
    .agg(
        n_cases=("case_submitter_id", "size"),
        n_leukocyte_fraction=("leukocyte_fraction", "count"),
    )
    .reset_index()
)

leukocyte_coverage_by_project["coverage"] = (
    leukocyte_coverage_by_project["n_leukocyte_fraction"]
    / leukocyte_coverage_by_project["n_cases"]
)

display(
    leukocyte_coverage_by_project.sort_values(
        "coverage"
    ).reset_index(drop=True)
)

,project_id,n_cases,n_leukocyte_fraction,coverage
0,TCGA-DLBC,48,0,0.000000
1,TCGA-LAML,134,0,0.000000
2,TCGA-THYM,120,0,0.000000
3,TCGA-TGCT,150,133,0.886667
4,TCGA-BRCA,1089,1074,0.986226
5,TCGA-CHOL,35,35,1.000000
6,TCGA-CESC,304,304,1.000000
7,TCGA-BLCA,405,405,1.000000
8,TCGA-ESCA,184,184,1.000000
9,TCGA-GBM,230,230,1.000000


In [35]:
# =============================================================================
# Summarize primary external covariates by TCGA project
# =============================================================================

external_covariate_summary = (
    confounder_table
    .melt(
        id_vars="project_id",
        value_vars=[
            "absolute_purity",
            "leukocyte_fraction",
        ],
        var_name="covariate",
        value_name="value",
    )
    .groupby(
        ["project_id", "covariate"],
        observed=True,
    )
    .agg(
        n_available=("value", "count"),
        median=("value", "median"),
        q1=("value", lambda values: values.quantile(0.25)),
        q3=("value", lambda values: values.quantile(0.75)),
    )
    .reset_index()
)

display(external_covariate_summary)

,project_id,covariate,n_available,median,q1,q3
0,TCGA-ACC,absolute_purity,77,0.850000,0.760000,0.910000
1,TCGA-ACC,leukocyte_fraction,79,0.055740,0.035022,0.153629
2,TCGA-BLCA,absolute_purity,394,0.610000,0.422500,0.800000
3,TCGA-BLCA,leukocyte_fraction,405,0.201592,0.107030,0.339615
4,TCGA-BRCA,absolute_purity,1041,0.590000,0.460000,0.740000
...,...,...,...,...,...,...
61,TCGA-UCEC,leukocyte_fraction,543,0.129028,0.076621,0.213691
62,TCGA-UCS,absolute_purity,56,0.855000,0.757500,0.930000
63,TCGA-UCS,leukocyte_fraction,57,0.061868,0.031533,0.137671
64,TCGA-UVM,absolute_purity,80,0.950000,0.907500,0.990000


In [36]:
# =============================================================================
# Assess within-project purity–leukocyte associations
# =============================================================================

purity_leukocyte_association = (
    confounder_table
    .groupby("project_id", observed=True)
    .apply(
        lambda group: pd.Series(
            {
                "n_complete": group[
                    ["absolute_purity", "leukocyte_fraction"]
                ].dropna().shape[0],
                "spearman_rho": group[
                    "absolute_purity"
                ].corr(
                    group["leukocyte_fraction"],
                    method="spearman",
                ),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

display(
    purity_leukocyte_association.sort_values(
        "spearman_rho"
    ).reset_index(drop=True)
)

,project_id,n_complete,spearman_rho
0,TCGA-TGCT,133.0,-0.928776
1,TCGA-CHOL,35.0,-0.915575
2,TCGA-KIRP,276.0,-0.880274
3,TCGA-KICH,66.0,-0.876341
4,TCGA-SKCM,103.0,-0.869853
5,TCGA-BLCA,394.0,-0.849924
6,TCGA-MESO,81.0,-0.845540
7,TCGA-SARC,242.0,-0.844310
8,TCGA-UCS,56.0,-0.841918
9,TCGA-GBM,221.0,-0.835539


In [37]:
print(
    purity_leukocyte_association.sort_values(
        "spearman_rho"
    ).reset_index(drop=True)
)

   project_id  n_complete  spearman_rho
0   TCGA-TGCT       133.0     -0.928776
1   TCGA-CHOL        35.0     -0.915575
2   TCGA-KIRP       276.0     -0.880274
3   TCGA-KICH        66.0     -0.876341
4   TCGA-SKCM       103.0     -0.869853
5   TCGA-BLCA       394.0     -0.849924
6   TCGA-MESO        81.0     -0.845540
7   TCGA-SARC       242.0     -0.844310
8    TCGA-UCS        56.0     -0.841918
9    TCGA-GBM       221.0     -0.835539
10  TCGA-CESC       291.0     -0.831158
11  TCGA-ESCA       161.0     -0.824477
12  TCGA-COAD       441.0     -0.804613
13   TCGA-ACC        77.0     -0.803726
14    TCGA-OV       409.0     -0.800513
15  TCGA-LIHC       358.0     -0.792538
16  TCGA-READ       161.0     -0.779787
17  TCGA-LUAD       499.0     -0.771625
18  TCGA-HNSC       507.0     -0.767916
19  TCGA-STAD       400.0     -0.755531
20  TCGA-UCEC       527.0     -0.741764
21  TCGA-BRCA      1028.0     -0.739691
22   TCGA-UVM        80.0     -0.737777
23  TCGA-LUSC       492.0     -0.728675


In [38]:
# =============================================================================
# Summarize biological covariates by methylation platform
# =============================================================================

mixed_projects = set(
    project_platform_summary.loc[
        project_platform_summary["mixed_platform"],
        "project_id",
    ]
)

platform_biological_summary = (
    confounder_table[
        confounder_table["project_id"].isin(mixed_projects)
    ]
    .groupby(
        ["project_id", "methylation_platform"],
        observed=True,
    )
    .agg(
        n_cases=("case_submitter_id", "size"),
        n_absolute=("absolute_purity", "count"),
        median_absolute_purity=("absolute_purity", "median"),
        n_leukocyte=("leukocyte_fraction", "count"),
        median_leukocyte_fraction=("leukocyte_fraction", "median"),
    )
    .reset_index()
)

display(platform_biological_summary)

,project_id,methylation_platform,n_cases,n_absolute,median_absolute_purity,n_leukocyte,median_leukocyte_fraction
0,TCGA-BRCA,Illumina Human Methylation 27,312,298,0.640,312,0.157680
1,TCGA-BRCA,Illumina Human Methylation 450,777,743,0.570,762,0.193935
2,TCGA-COAD,Illumina Human Methylation 27,160,153,0.690,160,0.141694
3,TCGA-COAD,Illumina Human Methylation 450,293,288,0.645,293,0.181931
4,TCGA-GBM,Illumina Human Methylation 27,150,145,0.780,150,0.137706
5,TCGA-GBM,Illumina Human Methylation 450,80,76,0.790,80,0.119411
6,TCGA-KIRC,Illumina Human Methylation 27,169,151,0.540,169,0.208490
7,TCGA-KIRC,Illumina Human Methylation 450,318,304,0.560,318,0.231804
8,TCGA-KIRP,Illumina Human Methylation 27,6,6,0.720,6,0.160917
9,TCGA-KIRP,Illumina Human Methylation 450,274,270,0.775,274,0.179933


In [39]:
# =============================================================================
# Summarize within-project methylation-platform contrasts
# =============================================================================

platform_contrasts = (
    platform_biological_summary
    .assign(
        platform=lambda table: table["methylation_platform"].replace(
            {
                "Illumina Human Methylation 27": "HM27",
                "Illumina Human Methylation 450": "HM450",
            }
        )
    )
    .pivot(
        index="project_id",
        columns="platform",
        values=[
            "n_cases",
            "median_absolute_purity",
            "median_leukocyte_fraction",
        ],
    )
)

platform_contrasts["minimum_platform_n"] = (
    platform_contrasts["n_cases"].min(axis=1)
)

platform_contrasts["purity_difference_hm450_minus_hm27"] = (
    platform_contrasts["median_absolute_purity"]["HM450"]
    - platform_contrasts["median_absolute_purity"]["HM27"]
)

platform_contrasts["leukocyte_difference_hm450_minus_hm27"] = (
    platform_contrasts["median_leukocyte_fraction"]["HM450"]
    - platform_contrasts["median_leukocyte_fraction"]["HM27"]
)

display(
    platform_contrasts[
        [
            "minimum_platform_n",
            "purity_difference_hm450_minus_hm27",
            "leukocyte_difference_hm450_minus_hm27",
        ]
    ].sort_values(
        "leukocyte_difference_hm450_minus_hm27"
    )
)

,minimum_platform_n,purity_difference_hm450_minus_hm27,leukocyte_difference_hm450_minus_hm27
platform,,,
project_id,,,
TCGA-OV,9.0,0.050,-0.065458
TCGA-GBM,80.0,0.010,-0.018295
TCGA-UCEC,115.0,-0.015,0.001912
TCGA-LUAD,59.0,-0.020,0.007080
TCGA-KIRP,6.0,0.055,0.019016
TCGA-KIRC,169.0,0.020,0.023314
TCGA-BRCA,312.0,-0.070,0.036255
TCGA-COAD,160.0,-0.045,0.040238


In [40]:
# =============================================================================
# Derive TCGA batch-associated variables
# =============================================================================

batch_covariates = final_sample_mapping[
    [
        "sample_submitter_id",
        "case_submitter_id",
        "rna_aliquot_submitter_id",
        "methylation_aliquot_submitter_id",
    ]
].copy()

batch_covariates["tissue_source_site"] = (
    batch_covariates["case_submitter_id"]
    .str.split("-")
    .str[1]
)

batch_covariates["rna_plate"] = (
    batch_covariates["rna_aliquot_submitter_id"]
    .str.split("-")
    .str[-2]
)

batch_covariates["rna_center"] = (
    batch_covariates["rna_aliquot_submitter_id"]
    .str.split("-")
    .str[-1]
)

batch_covariates["methylation_plate"] = (
    batch_covariates["methylation_aliquot_submitter_id"]
    .str.split("-")
    .str[-2]
)

batch_covariates["methylation_center"] = (
    batch_covariates["methylation_aliquot_submitter_id"]
    .str.split("-")
    .str[-1]
)

confounder_table = confounder_table.merge(
    batch_covariates.drop(
        columns=[
            "case_submitter_id",
            "rna_aliquot_submitter_id",
            "methylation_aliquot_submitter_id",
        ]
    ),
    on="sample_submitter_id",
    how="left",
)

display(
    confounder_table[
        [
            "sample_submitter_id",
            "tissue_source_site",
            "rna_plate",
            "rna_center",
            "methylation_plate",
            "methylation_center",
        ]
    ].head()
)

,sample_submitter_id,tissue_source_site,rna_plate,rna_center,methylation_plate,methylation_center
0,TCGA-02-0003-01A,02,A96R,41,0186,05
1,TCGA-02-0033-01A,02,A96R,41,0186,05
2,TCGA-02-0038-01A,02,A96R,41,0186,05
3,TCGA-02-0047-01A,02,1849,01,0186,05
4,TCGA-02-0055-01A,02,1849,01,0186,05


In [41]:
# =============================================================================
# Summarize batch-associated variable cardinality
# =============================================================================

batch_columns = [
    "tissue_source_site",
    "rna_plate",
    "rna_center",
    "methylation_plate",
    "methylation_center",
]

batch_level_summary = pd.DataFrame(
    {
        "covariate": batch_columns,
        "n_levels": [
            confounder_table[column].nunique()
            for column in batch_columns
        ],
        "n_singleton_levels": [
            confounder_table[column]
            .value_counts()
            .eq(1)
            .sum()
            for column in batch_columns
        ],
        "median_cases_per_level": [
            confounder_table[column]
            .value_counts()
            .median()
            for column in batch_columns
        ],
    }
)

display(batch_level_summary)

,covariate,n_levels,n_singleton_levels,median_cases_per_level
0,tissue_source_site,702,152,5.5
1,rna_plate,238,4,39.0
2,rna_center,5,0,388.0
3,methylation_plate,265,2,36.0
4,methylation_center,1,0,9965.0


In [42]:
# =============================================================================
# Summarize within-project batch structure
# =============================================================================

batch_support_by_project = (
    confounder_table
    .groupby("project_id", observed=True)
    .agg(
        n_cases=("case_submitter_id", "size"),
        n_rna_centers=("rna_center", "nunique"),
        n_rna_plates=("rna_plate", "nunique"),
        n_methylation_plates=("methylation_plate", "nunique"),
        n_tissue_source_sites=("tissue_source_site", "nunique"),
    )
    .reset_index()
)

display(batch_support_by_project)

,project_id,n_cases,n_rna_centers,n_rna_plates,n_methylation_plates,n_tissue_source_sites
0,TCGA-ACC,79,1,1,1,5
1,TCGA-BLCA,405,1,28,25,35
2,TCGA-BRCA,1089,1,40,36,40
3,TCGA-CESC,304,1,21,19,31
4,TCGA-CHOL,35,1,1,1,9
5,TCGA-COAD,453,1,23,20,25
6,TCGA-DLBC,48,1,5,5,8
7,TCGA-ESCA,184,1,10,10,19
8,TCGA-GBM,230,2,7,15,13
9,TCGA-HNSC,520,1,19,19,28


In [43]:
# =============================================================================
# Summarize within-project plate support
# =============================================================================

plate_level_counts = (
    confounder_table
    .melt(
        id_vars=["project_id", "case_submitter_id"],
        value_vars=[
            "rna_plate",
            "methylation_plate",
        ],
        var_name="plate_variable",
        value_name="plate",
    )
    .groupby(
        ["project_id", "plate_variable", "plate"],
        observed=True,
    )
    .size()
    .reset_index(name="n_cases")
)

plate_support_summary = (
    plate_level_counts
    .groupby(
        ["project_id", "plate_variable"],
        observed=True,
    )
    .agg(
        n_plates=("plate", "nunique"),
        minimum_cases_per_plate=("n_cases", "min"),
        median_cases_per_plate=("n_cases", "median"),
        maximum_cases_per_plate=("n_cases", "max"),
        n_singleton_plates=(
            "n_cases",
            lambda values: values.eq(1).sum(),
        ),
        n_plates_below_five_cases=(
            "n_cases",
            lambda values: values.lt(5).sum(),
        ),
    )
    .reset_index()
)

display(plate_support_summary)

,project_id,plate_variable,n_plates,minimum_cases_per_plate,median_cases_per_plate,maximum_cases_per_plate,n_singleton_plates,n_plates_below_five_cases
0,TCGA-ACC,methylation_plate,1,79,79.0,79,0,0
1,TCGA-ACC,rna_plate,1,79,79.0,79,0,0
2,TCGA-BLCA,methylation_plate,25,1,12.0,56,1,2
3,TCGA-BLCA,rna_plate,28,1,11.0,56,4,5
4,TCGA-BRCA,methylation_plate,36,8,19.0,93,0,0
...,...,...,...,...,...,...,...,...
61,TCGA-UCEC,rna_plate,28,2,17.5,49,0,2
62,TCGA-UCS,methylation_plate,1,57,57.0,57,0,0
63,TCGA-UCS,rna_plate,1,57,57.0,57,0,0
64,TCGA-UVM,methylation_plate,1,80,80.0,80,0,0


In [44]:
# =============================================================================
# Summarize RNA-seq technical variation across plates
# =============================================================================

rna_plate_metrics = (
    confounder_table
    .groupby(
        ["project_id", "rna_plate"],
        observed=True,
    )
    .agg(
        n_cases=("case_submitter_id", "size"),
        median_assigned_fraction=(
            "gene_assigned_fraction_of_accounted",
            "median",
        ),
    )
    .reset_index()
)

rna_plate_variation = (
    rna_plate_metrics
    .groupby("project_id", observed=True)
    .agg(
        n_rna_plates=("rna_plate", "nunique"),
        minimum_plate_n=("n_cases", "min"),
        median_plate_assigned_fraction=(
            "median_assigned_fraction",
            "median",
        ),
        minimum_plate_assigned_fraction=(
            "median_assigned_fraction",
            "min",
        ),
        maximum_plate_assigned_fraction=(
            "median_assigned_fraction",
            "max",
        ),
    )
    .reset_index()
)

rna_plate_variation["assigned_fraction_range"] = (
    rna_plate_variation["maximum_plate_assigned_fraction"]
    - rna_plate_variation["minimum_plate_assigned_fraction"]
)

display(
    rna_plate_variation.sort_values(
        "assigned_fraction_range",
        ascending=False,
    ).reset_index(drop=True)
)

,project_id,n_rna_plates,minimum_plate_n,median_plate_assigned_fraction,minimum_plate_assigned_fraction,maximum_plate_assigned_fraction,assigned_fraction_range
0,TCGA-LUAD,22,1,0.754174,0.332517,0.776412,0.443896
1,TCGA-STAD,18,7,0.612308,0.313019,0.714701,0.401683
2,TCGA-GBM,7,1,0.472958,0.424631,0.742871,0.318241
3,TCGA-OV,9,1,0.657386,0.440151,0.684159,0.244008
4,TCGA-LAML,5,1,0.410661,0.286956,0.476149,0.189193
5,TCGA-THCA,18,3,0.775007,0.634916,0.788320,0.153403
6,TCGA-BRCA,40,1,0.771367,0.686801,0.792480,0.105679
7,TCGA-COAD,23,3,0.754709,0.670709,0.776295,0.105586
8,TCGA-SARC,16,1,0.779955,0.695008,0.794922,0.099914
9,TCGA-SKCM,16,2,0.768877,0.690606,0.790162,0.099556


In [45]:
# =============================================================================
# Summarize RNA-seq variation across supported plates
# =============================================================================

supported_rna_plate_metrics = rna_plate_metrics[
    rna_plate_metrics["n_cases"].ge(5)
].copy()

supported_rna_plate_variation = (
    supported_rna_plate_metrics
    .groupby("project_id", observed=True)
    .agg(
        n_supported_plates=("rna_plate", "nunique"),
        n_cases_on_supported_plates=("n_cases", "sum"),
        minimum_plate_median=(
            "median_assigned_fraction",
            "min",
        ),
        maximum_plate_median=(
            "median_assigned_fraction",
            "max",
        ),
    )
    .reset_index()
)

supported_rna_plate_variation["supported_plate_range"] = (
    supported_rna_plate_variation["maximum_plate_median"]
    - supported_rna_plate_variation["minimum_plate_median"]
)

supported_rna_plate_variation.loc[
    supported_rna_plate_variation["n_supported_plates"].lt(2),
    "supported_plate_range",
] = np.nan

display(
    supported_rna_plate_variation.sort_values(
        "supported_plate_range",
        ascending=False,
    ).reset_index(drop=True)
)

,project_id,n_supported_plates,n_cases_on_supported_plates,minimum_plate_median,maximum_plate_median,supported_plate_range
0,TCGA-STAD,18,412,0.313019,0.714701,0.401683
1,TCGA-GBM,6,229,0.424631,0.707593,0.282962
2,TCGA-LAML,4,133,0.352484,0.476149,0.123665
3,TCGA-UCEC,26,537,0.688979,0.784044,0.095065
4,TCGA-ESCA,10,184,0.607161,0.692499,0.085338
5,TCGA-DLBC,4,47,0.671408,0.750975,0.079568
6,TCGA-BLCA,23,398,0.710995,0.787330,0.076335
7,TCGA-KIRC,14,482,0.715742,0.791800,0.076058
8,TCGA-PRAD,18,494,0.710473,0.785730,0.075257
9,TCGA-SKCM,9,78,0.723946,0.787888,0.063942


In [46]:
# =============================================================================
# Summarize methylation variation across supported plates
# =============================================================================

methylation_plate_metrics = (
    confounder_table
    .groupby(
        [
            "project_id",
            "methylation_platform",
            "methylation_plate",
        ],
        observed=True,
    )
    .agg(
        n_cases=("case_submitter_id", "size"),
        median_missing_beta_fraction=(
            "missing_beta_fraction",
            "median",
        ),
    )
    .reset_index()
)

supported_methylation_plate_metrics = (
    methylation_plate_metrics[
        methylation_plate_metrics["n_cases"].ge(5)
    ]
    .copy()
)

supported_methylation_plate_variation = (
    supported_methylation_plate_metrics
    .groupby(
        ["project_id", "methylation_platform"],
        observed=True,
    )
    .agg(
        n_supported_plates=("methylation_plate", "nunique"),
        n_cases_on_supported_plates=("n_cases", "sum"),
        minimum_plate_median=(
            "median_missing_beta_fraction",
            "min",
        ),
        maximum_plate_median=(
            "median_missing_beta_fraction",
            "max",
        ),
    )
    .reset_index()
)

supported_methylation_plate_variation[
    "supported_plate_range"
] = (
    supported_methylation_plate_variation[
        "maximum_plate_median"
    ]
    - supported_methylation_plate_variation[
        "minimum_plate_median"
    ]
)

supported_methylation_plate_variation.loc[
    supported_methylation_plate_variation[
        "n_supported_plates"
    ].lt(2),
    "supported_plate_range",
] = np.nan

display(
    supported_methylation_plate_variation.sort_values(
        "supported_plate_range",
        ascending=False,
    ).reset_index(drop=True)
)

,project_id,methylation_platform,n_supported_plates,n_cases_on_supported_plates,minimum_plate_median,maximum_plate_median,supported_plate_range
0,TCGA-LUSC,Illumina Human Methylation 450,17,370,0.138829,0.218890,0.080061
1,TCGA-STAD,Illumina Human Methylation 450,16,373,0.137074,0.202448,0.065374
2,TCGA-BRCA,Illumina Human Methylation 450,32,777,0.136138,0.199894,0.063757
3,TCGA-LIHC,Illumina Human Methylation 450,18,363,0.138936,0.200199,0.061263
4,TCGA-LUAD,Illumina Human Methylation 450,18,453,0.137215,0.197068,0.059853
5,TCGA-SARC,Illumina Human Methylation 450,15,259,0.142412,0.200889,0.058477
6,TCGA-CESC,Illumina Human Methylation 450,18,302,0.138775,0.195343,0.056568
7,TCGA-PAAD,Illumina Human Methylation 450,11,178,0.140525,0.196781,0.056256
8,TCGA-UCEC,Illumina Human Methylation 450,21,422,0.136684,0.188941,0.052257
9,TCGA-SKCM,Illumina Human Methylation 450,10,82,0.139828,0.191433,0.051605


In [47]:
# =============================================================================
# PanImmune proliferation-resource paths
# =============================================================================

PANIMMUNE_GENE_SET_DEFINITIONS_PATH = (
    CONFOUNDER_RAW_DIR
    / "PanImmune_GeneSet_Definitions.xlsx"
)

PANIMMUNE_SIGNATURE_SCORES_PATH = (
    CONFOUNDER_RAW_DIR
    / "Scores_160_Signatures.tsv"
)

In [48]:
# =============================================================================
# Load the PanImmune proliferation gene-set definition
# =============================================================================

panimmune_gene_sets = pd.read_excel(
    PANIMMUNE_GENE_SET_DEFINITIONS_PATH,
    sheet_name="Genes",
)

proliferation_gene_set = panimmune_gene_sets[
    panimmune_gene_sets["SetName"].eq(
        "Module11_Prolif_score"
    )
].copy()

print(
    "Proliferation gene-set rows: "
    f"{len(proliferation_gene_set):,}"
)
print(
    "Unique proliferation genes: "
    f"{proliferation_gene_set['Gene'].nunique():,}"
)

display(proliferation_gene_set.head())

Proliferation gene-set rows: 120
Unique proliferation genes: 120


,SetName,Gene
488,Module11_Prolif_score,CDKN3
489,Module11_Prolif_score,NDC80
490,Module11_Prolif_score,RNASEH2A
491,Module11_Prolif_score,CENPA
492,Module11_Prolif_score,SMC2


In [49]:
# =============================================================================
# Inspect PanImmune precomputed signature scores
# =============================================================================

panimmune_signature_scores = pd.read_csv(
    PANIMMUNE_SIGNATURE_SCORES_PATH,
    sep="\t",
    low_memory=False,
)

print(
    "PanImmune signature-score shape: "
    f"{panimmune_signature_scores.shape}"
)
print(
    "First columns:",
    panimmune_signature_scores.columns[:8].tolist(),
)

display(
    panimmune_signature_scores.iloc[:5, :8]
)

PanImmune signature-score shape: (160, 9131)
First columns: ['Source', 'SetName', 'TCGA-02-0047-01A-01R-1849-01', 'TCGA-02-0055-01A-01R-1849-01', 'TCGA-02-2483-01A-01R-1849-01', 'TCGA-02-2485-01A-01R-1849-01', 'TCGA-02-2486-01A-01R-1849-01', 'TCGA-04-1348-01A-01R-1565-13']


,Source,SetName,TCGA-02-0047-01A-01R-1849-01,TCGA-02-0055-01A-01R-1849-01,TCGA-02-2483-01A-01R-1849-01,TCGA-02-2485-01A-01R-1849-01,TCGA-02-2486-01A-01R-1849-01,TCGA-04-1348-01A-01R-1565-13
0,Senbabaoglu,Angiogenesis,0.192506,0.098558,0.202722,0.191013,0.056185,-0.043404
1,Senbabaoglu,APM1,0.449286,0.467428,0.438052,0.463771,0.476054,0.421535
2,Senbabaoglu,APM2,0.243735,0.299398,0.265885,0.171171,0.285493,0.346336
3,Wolf,ICS5_score,-1.519200,0.617800,-2.013000,-0.686000,-0.912000,1.694600
4,Wolf,LIexpression_score,-1.858600,-0.738400,-1.288100,-1.105700,-0.126300,0.767900


In [50]:
# =============================================================================
# Extract the PanImmune proliferation score
# =============================================================================

proliferation_score_row = (
    panimmune_signature_scores.loc[
        panimmune_signature_scores["SetName"].eq(
            "Module11_Prolif_score"
        )
    ]
    .drop(columns=["Source", "SetName"])
    .iloc[0]
)

panimmune_proliferation_scores = (
    proliferation_score_row
    .rename("panimmune_proliferation_score")
    .rename_axis("rna_aliquot_submitter_id")
    .reset_index()
)

print(
    "PanImmune proliferation-score records: "
    f"{len(panimmune_proliferation_scores):,}"
)
print(
    "Non-missing scores: "
    f"{panimmune_proliferation_scores['panimmune_proliferation_score'].notna().sum():,}"
)

display(panimmune_proliferation_scores.head())

PanImmune proliferation-score records: 9,129
Non-missing scores: 9,129


,rna_aliquot_submitter_id,panimmune_proliferation_score
0,TCGA-02-0047-01A-01R-1849-01,-0.0879
1,TCGA-02-0055-01A-01R-1849-01,-0.0651
2,TCGA-02-2483-01A-01R-1849-01,0.8762
3,TCGA-02-2485-01A-01R-1849-01,0.3223
4,TCGA-02-2486-01A-01R-1849-01,-1.2546


In [51]:
# =============================================================================
# Assess PanImmune proliferation-score coverage
# =============================================================================

proliferation_score_aliquots = set(
    panimmune_proliferation_scores[
        "rna_aliquot_submitter_id"
    ]
)

proliferation_score_samples = set(
    panimmune_proliferation_scores[
        "rna_aliquot_submitter_id"
    ].str[:16]
)

exact_aliquot_match = (
    final_sample_mapping["rna_aliquot_submitter_id"]
    .isin(proliferation_score_aliquots)
)

same_sample_available = (
    final_sample_mapping["sample_submitter_id"]
    .isin(proliferation_score_samples)
)

print(
    "Exact RNA-aliquot matches: "
    f"{exact_aliquot_match.sum():,}"
)
print(
    "Same sample, different RNA aliquot only: "
    f"{(same_sample_available & ~exact_aliquot_match).sum():,}"
)
print(
    "Sample absent from PanImmune scores: "
    f"{(~same_sample_available).sum():,}"
)
print(
    "Exact-aliquot coverage: "
    f"{exact_aliquot_match.mean():.2%}"
)

Exact RNA-aliquot matches: 9,006
Same sample, different RNA aliquot only: 5
Sample absent from PanImmune scores: 954
Exact-aliquot coverage: 90.38%


In [52]:
# =============================================================================
# Add exact-aliquot PanImmune proliferation scores
# =============================================================================

proliferation_for_merge = (
    final_sample_mapping[
        [
            "sample_submitter_id",
            "rna_aliquot_submitter_id",
        ]
    ]
    .merge(
        panimmune_proliferation_scores,
        on="rna_aliquot_submitter_id",
        how="left",
    )
    [
        [
            "sample_submitter_id",
            "panimmune_proliferation_score",
        ]
    ]
)

confounder_table = confounder_table.merge(
    proliferation_for_merge,
    on="sample_submitter_id",
    how="left",
)

print(
    "Cases with PanImmune proliferation score: "
    f"{confounder_table['panimmune_proliferation_score'].notna().sum():,}"
)
print(
    "Cases without PanImmune proliferation score: "
    f"{confounder_table['panimmune_proliferation_score'].isna().sum():,}"
)

Cases with PanImmune proliferation score: 9,006
Cases without PanImmune proliferation score: 959


In [53]:
# =============================================================================
# Summarize proliferation-score coverage by TCGA project
# =============================================================================

proliferation_coverage_by_project = (
    confounder_table
    .groupby("project_id", observed=True)
    .agg(
        n_cases=("case_submitter_id", "size"),
        n_proliferation_score=(
            "panimmune_proliferation_score",
            "count",
        ),
    )
    .reset_index()
)

proliferation_coverage_by_project["coverage"] = (
    proliferation_coverage_by_project[
        "n_proliferation_score"
    ]
    / proliferation_coverage_by_project["n_cases"]
)

display(
    proliferation_coverage_by_project.sort_values(
        "coverage"
    ).reset_index(drop=True)
)

,project_id,n_cases,n_proliferation_score,coverage
0,TCGA-DLBC,48,0,0.000000
1,TCGA-LAML,134,0,0.000000
2,TCGA-THYM,120,0,0.000000
3,TCGA-GBM,230,121,0.526087
4,TCGA-OV,422,266,0.630332
5,TCGA-PRAD,496,404,0.814516
6,TCGA-PAAD,178,151,0.848315
7,TCGA-SARC,259,223,0.861004
8,TCGA-LUAD,512,454,0.886719
9,TCGA-READ,164,154,0.939024


In [54]:
# =============================================================================
# Assess within-project proliferation associations
# =============================================================================

proliferation_associations = (
    confounder_table
    .groupby("project_id", observed=True)
    .apply(
        lambda group: pd.Series(
            {
                "n_proliferation": (
                    group["panimmune_proliferation_score"]
                    .notna()
                    .sum()
                ),
                "rho_with_absolute_purity": (
                    group["panimmune_proliferation_score"]
                    .corr(
                        group["absolute_purity"],
                        method="spearman",
                    )
                ),
                "rho_with_leukocyte_fraction": (
                    group["panimmune_proliferation_score"]
                    .corr(
                        group["leukocyte_fraction"],
                        method="spearman",
                    )
                ),
                "rho_with_rna_assigned_fraction": (
                    group["panimmune_proliferation_score"]
                    .corr(
                        group[
                            "gene_assigned_fraction_of_accounted"
                        ],
                        method="spearman",
                    )
                ),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

display(
    proliferation_associations.sort_values(
        "rho_with_leukocyte_fraction"
    ).reset_index(drop=True)
)

,project_id,n_proliferation,rho_with_absolute_purity,rho_with_leukocyte_fraction,rho_with_rna_assigned_fraction
0,TCGA-GBM,121.0,0.553199,-0.487014,0.023235
1,TCGA-ACC,78.0,0.355930,-0.478825,0.192706
2,TCGA-STAD,387.0,0.388971,-0.348979,0.143108
3,TCGA-LUSC,485.0,0.435617,-0.322864,-0.003027
4,TCGA-TGCT,149.0,0.223755,-0.306930,0.130238
5,TCGA-SARC,223.0,0.224779,-0.263416,0.150247
6,TCGA-ESCA,173.0,0.334140,-0.261096,0.184915
7,TCGA-HNSC,514.0,0.331656,-0.228535,-0.018437
8,TCGA-CESC,300.0,0.269846,-0.228091,0.149016
9,TCGA-UCS,57.0,0.213080,-0.220638,0.184146


In [55]:
# =============================================================================
# Prepare final confounder-covariate table
# =============================================================================

final_confounder_covariates = confounder_table[
    [
        "final_sample_column_index",
        "case_submitter_id",
        "sample_submitter_id",
        "project_id",
        "methylation_platform",
        "absolute_call_status",
        "absolute_purity",
        "consensus_purity_estimate",
        "purity_availability",
        "leukocyte_fraction",
        "panimmune_proliferation_score",
        "gene_assigned_fraction_of_accounted",
        "missing_beta_fraction",
        "tissue_source_site",
        "rna_plate",
        "rna_center",
        "methylation_plate",
    ]
].copy()

print(
    "Final confounder-covariate table: "
    f"{final_confounder_covariates.shape}"
)

display(final_confounder_covariates.head())

Final confounder-covariate table: (9965, 17)


,final_sample_column_index,case_submitter_id,sample_submitter_id,project_id,methylation_platform,absolute_call_status,absolute_purity,consensus_purity_estimate,purity_availability,leukocyte_fraction,panimmune_proliferation_score,gene_assigned_fraction_of_accounted,missing_beta_fraction,tissue_source_site,rna_plate,rna_center,methylation_plate
0,0,TCGA-02-0003,TCGA-02-0003-01A,TCGA-GBM,Illumina Human Methylation 27,called,0.90,0.9850,CPE and ABSOLUTE,0.047688,NaN,0.439303,0.175792,02,A96R,41,0186
1,1,TCGA-02-0033,TCGA-02-0033-01A,TCGA-GBM,Illumina Human Methylation 27,called,0.45,0.7703,CPE and ABSOLUTE,0.614121,NaN,0.513737,0.099463,02,A96R,41,0186
2,2,TCGA-02-0038,TCGA-02-0038-01A,TCGA-GBM,Illumina Human Methylation 27,legacy_call,0.78,0.9362,CPE and ABSOLUTE,0.077622,NaN,0.325748,0.113387,02,A96R,41,0186
3,3,TCGA-02-0047,TCGA-02-0047-01A,TCGA-GBM,Illumina Human Methylation 27,called,0.69,0.8266,CPE and ABSOLUTE,0.235745,-0.0879,0.669034,0.140039,02,1849,01,0186
4,4,TCGA-02-0055,TCGA-02-0055-01A,TCGA-GBM,Illumina Human Methylation 27,called,0.55,0.6655,CPE and ABSOLUTE,0.298776,-0.0651,0.722150,0.110740,02,1849,01,0186


In [56]:
# =============================================================================
# Clarify external proliferation-score provenance
# =============================================================================

final_confounder_covariates = (
    final_confounder_covariates.rename(
        columns={
            "panimmune_proliferation_score": (
                "external_panimmune_proliferation_score"
            )
        }
    )
)

In [57]:
# =============================================================================
# Save final confounder-covariate table
# =============================================================================

CONFOUNDER_COVARIATES_OUTPUT_PATH = (
    Paths.metadata
    / "tcga_primary_tumor_multiomic_confounder_covariates.csv"
)

final_confounder_covariates.to_csv(
    CONFOUNDER_COVARIATES_OUTPUT_PATH,
    index=False,
)

print("Confounder-covariate table saved.")
print(f"Path: {project_relative_path(CONFOUNDER_COVARIATES_OUTPUT_PATH)}")
print(f"Shape: {final_confounder_covariates.shape}")

Confounder-covariate table saved.
Path: data/interim/metadata/tcga_primary_tumor_multiomic_confounder_covariates.csv
Shape: (9965, 17)


## Demographic-sex extension

Program discovery in notebook 205 identified candidate methylation components
with substantial chrX loading enrichment, including one component with nearly
exclusive chrX concentration.

This extension retrieves case-level demographic sex from GDC to support an
explicit sensitivity assessment of potential sex-chromosome confounding.

The existing confounder analyses are not repeated. The previously published
covariate artifact is loaded and extended without changing the frozen
9,965-case cohort.

In [58]:
# =======================================================
# Prepare demographic-sex extension
# =======================================================

from datetime import datetime, timezone
import json

import pandas as pd
import requests

from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)


CONFOUNDER_COVARIATES_OUTPUT_PATH = (
    Paths.metadata
    / "tcga_primary_tumor_multiomic_confounder_covariates.csv"
)

GDC_CASES_ENDPOINT = "https://api.gdc.cancer.gov/cases"

gdc_retrieval_date = (
    datetime.now(timezone.utc)
    .date()
    .isoformat()
)

GDC_DEMOGRAPHIC_RAW_PATH = (
    Paths.tcga
    / "confounders"
    / f"gdc_tcga_case_demographics_{gdc_retrieval_date}.json"
)

GDC_DEMOGRAPHIC_TABLE_PATH = (
    Paths.metadata
    / "tcga_primary_tumor_case_demographics.csv"
)

final_confounder_covariates = pd.read_csv(
    CONFOUNDER_COVARIATES_OUTPUT_PATH
)

frozen_case_ids = (
    final_confounder_covariates[
        "case_submitter_id"
    ]
    .drop_duplicates()
    .to_list()
)

print(
    "Existing confounder-covariate table: "
    f"{final_confounder_covariates.shape}"
)
print(f"Frozen case IDs: {len(frozen_case_ids):,}")
print(
    "Raw demographic output: "
    f"{project_relative_path(GDC_DEMOGRAPHIC_RAW_PATH)}"
)
print(
    "Interim demographic table: "
    f"{project_relative_path(GDC_DEMOGRAPHIC_TABLE_PATH)}"
)

Existing confounder-covariate table: (9965, 17)
Frozen case IDs: 9,965
Raw demographic output: data/raw/tcga/confounders/gdc_tcga_case_demographics_2026-08-06.json
Interim demographic table: data/interim/metadata/tcga_primary_tumor_case_demographics.csv


In [60]:
# =======================================================
# Test GDC API connectivity with retries
# =======================================================

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


gdc_session = requests.Session()

gdc_retry_policy = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=2,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504,
    ],
    allowed_methods=["GET"],
)

gdc_session.mount(
    "https://",
    HTTPAdapter(
        max_retries=gdc_retry_policy
    ),
)

gdc_status_response = gdc_session.get(
    "https://api.gdc.cancer.gov/status",
    headers={
        "User-Agent": (
            "pancancer-epigenetics/204 "
            "(TCGA demographic metadata retrieval)"
        ),
    },
    timeout=(30, 120),
)

gdc_status_response.raise_for_status()

{
    "status_code": gdc_status_response.status_code,
    "response": gdc_status_response.json(),
}

{'status_code': 200,
 'response': {'commit': '8f7c2a51ab0084b216ad1b62a3fae8b945439c53',
  'data_release': 'Data Release 45.0 - December 04, 2025',
  'data_release_version': {'major': 45,
   'minor': 0,
   'release_date': '2025-12-04'},
  'status': 'OK',
  'tag': '8.5.0',
  'version': 1}}

In [61]:
# =======================================================
# Test GDC demographic retrieval for one case
# =======================================================

test_case_submitter_id = frozen_case_ids[0]

test_query_params = {
    "filters": json.dumps(
        {
            "op": "in",
            "content": {
                "field": "submitter_id",
                "value": [
                    test_case_submitter_id
                ],
            },
        }
    ),
    "fields": (
        "submitter_id,"
        "demographic.sex_at_birth"
    ),
    "expand": "demographic",
    "format": "JSON",
    "size": 1,
}

test_response = gdc_session.get(
    GDC_CASES_ENDPOINT,
    params=test_query_params,
    headers={
        "User-Agent": (
            "pancancer-epigenetics/204 "
            "(TCGA demographic metadata retrieval)"
        ),
    },
    timeout=(30, 120),
)

test_response.raise_for_status()

test_response_body = test_response.json()

test_hits = (
    test_response_body
    .get("data", {})
    .get("hits", [])
)

{
    "requested_case": test_case_submitter_id,
    "status_code": test_response.status_code,
    "returned_hits": len(test_hits),
    "warnings": test_response_body.get(
        "warnings",
        {}
    ),
    "hits": test_hits,
}

{'requested_case': 'TCGA-02-0003',
 'status_code': 200,
 'returned_hits': 1,
 'warnings': {},
 'hits': [{'id': 'df3c1d61-79c1-43e9-971a-8029497ffeab',
   'submitter_id': 'TCGA-02-0003',
   'demographic': {'race': 'white',
    'ethnicity': 'not hispanic or latino',
    'vital_status': 'Dead',
    'age_at_index': 50,
    'submitter_id': 'TCGA-02-0003_demographic',
    'days_to_birth': -18341,
    'created_datetime': None,
    'year_of_birth': None,
    'demographic_id': 'b3d375be-0fe4-527a-8b7e-a537551a61a5',
    'updated_datetime': '2025-10-16T15:41:03.970912-05:00',
    'age_is_obfuscated': False,
    'days_to_death': 144,
    'state': 'released',
    'sex_at_birth': 'male',
    'year_of_death': None}}]}

In [62]:
# =======================================================
# Retrieve frozen-cohort demographic metadata
# =======================================================

full_retry_policy = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=2,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504,
    ],
    allowed_methods=[
        "GET",
        "POST",
    ],
)

full_gdc_session = requests.Session()

full_gdc_session.mount(
    "https://",
    HTTPAdapter(
        max_retries=full_retry_policy
    ),
)

full_query_payload = {
    "filters": {
        "op": "in",
        "content": {
            "field": "submitter_id",
            "value": frozen_case_ids,
        },
    },
    "fields": (
        "submitter_id,"
        "demographic.sex_at_birth"
    ),
    "expand": "demographic",
    "format": "JSON",
    "size": len(frozen_case_ids),
}

full_response = full_gdc_session.post(
    GDC_CASES_ENDPOINT,
    headers={
        "Content-Type": "application/json",
        "User-Agent": (
            "pancancer-epigenetics/204 "
            "(TCGA demographic metadata retrieval)"
        ),
    },
    json=full_query_payload,
    timeout=(30, 300),
)

full_response.raise_for_status()

full_response_body = full_response.json()

gdc_demographic_hits = (
    full_response_body
    .get("data", {})
    .get("hits", [])
)

GDC_DEMOGRAPHIC_RAW_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

raw_demographic_export = {
    "retrieved_at_utc": (
        datetime.now(timezone.utc)
        .isoformat()
    ),
    "endpoint": GDC_CASES_ENDPOINT,
    "gdc_status": (
        gdc_status_response.json()
    ),
    "requested_case_count": len(
        frozen_case_ids
    ),
    "response": full_response_body,
}

with GDC_DEMOGRAPHIC_RAW_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        raw_demographic_export,
        file,
        indent=2,
    )

returned_case_ids = [
    hit.get("submitter_id")
    for hit in gdc_demographic_hits
]

missing_case_ids = sorted(
    set(frozen_case_ids)
    - set(returned_case_ids)
)

sex_at_birth_counts = pd.Series(
    [
        (
            hit.get("demographic") or {}
        ).get("sex_at_birth")
        for hit in gdc_demographic_hits
    ],
    name="sex_at_birth",
).value_counts(
    dropna=False
)

print(f"Requested cases: {len(frozen_case_ids):,}")
print(f"Returned records: {len(gdc_demographic_hits):,}")
print(
    "Unique returned cases: "
    f"{len(set(returned_case_ids)):,}"
)
print(f"Missing frozen cases: {len(missing_case_ids):,}")
print(
    "Raw demographic response saved: "
    f"{project_relative_path(GDC_DEMOGRAPHIC_RAW_PATH)}"
)

display(sex_at_birth_counts)

Requested cases: 9,965
Returned records: 9,965
Unique returned cases: 9,965
Missing frozen cases: 0
Raw demographic response saved: data/raw/tcga/confounders/gdc_tcga_case_demographics_2026-08-06.json


sex_at_birth
female     5210
male       4749
unknown       4
NaN           2
Name: count, dtype: int64

In [63]:
# =======================================================
# Build case-level demographic table
# =======================================================

demographic_rows = []

for hit in gdc_demographic_hits:
    demographic = hit.get("demographic") or {}

    sex_at_birth = demographic.get(
        "sex_at_birth"
    )

    if isinstance(sex_at_birth, str):
        sex_at_birth = (
            sex_at_birth
            .strip()
            .lower()
        )
    else:
        sex_at_birth = pd.NA

    demographic_rows.append(
        {
            "case_submitter_id": hit.get(
                "submitter_id"
            ),
            "gdc_case_id": hit.get("id"),
            "sex_at_birth": sex_at_birth,
            "gdc_demographic_id": demographic.get(
                "demographic_id"
            ),
            "gdc_demographic_updated_datetime": (
                demographic.get(
                    "updated_datetime"
                )
            ),
        }
    )

gdc_case_demographics = pd.DataFrame(
    demographic_rows
)

frozen_case_order = {
    case_id: index
    for index, case_id in enumerate(
        frozen_case_ids
    )
}

gdc_case_demographics[
    "final_case_order"
] = gdc_case_demographics[
    "case_submitter_id"
].map(frozen_case_order)

gdc_case_demographics = (
    gdc_case_demographics
    .sort_values("final_case_order")
    .drop(columns="final_case_order")
    .reset_index(drop=True)
)

GDC_DEMOGRAPHIC_TABLE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

gdc_case_demographics.to_csv(
    GDC_DEMOGRAPHIC_TABLE_PATH,
    index=False,
)

print(
    "Case-level demographic table: "
    f"{gdc_case_demographics.shape}"
)
print(
    "Unique case IDs: "
    f"{gdc_case_demographics['case_submitter_id'].nunique():,}"
)
print(
    "Saved: "
    f"{project_relative_path(GDC_DEMOGRAPHIC_TABLE_PATH)}"
)

display(
    gdc_case_demographics[
        "sex_at_birth"
    ].value_counts(
        dropna=False
    )
)

gdc_case_demographics.head()

Case-level demographic table: (9965, 5)
Unique case IDs: 9,965
Saved: data/interim/metadata/tcga_primary_tumor_case_demographics.csv


sex_at_birth
female     5210
male       4749
unknown       4
NaN           2
Name: count, dtype: int64

,case_submitter_id,gdc_case_id,sex_at_birth,gdc_demographic_id,gdc_demographic_updated_datetime
0,TCGA-02-0003,df3c1d61-79c1-43e9-971a-8029497ffeab,male,b3d375be-0fe4-527a-8b7e-a537551a61a5,2025-10-16T15:41:03.970912-05:00
1,TCGA-02-0033,fc5ad666-d67a-4a5c-8e4e-1c8d099e9f85,male,94f90036-5a63-5b03-8a81-06cbef649614,2025-10-16T15:41:03.970912-05:00
2,TCGA-02-0038,fcce7f31-a392-4177-8dca-cfbe6e73fc2e,female,f22196b5-452f-5111-a1d6-41bbcbdd6ef6,2025-10-16T15:41:03.970912-05:00
3,TCGA-02-0047,3caf009f-d9e0-4c57-b1d9-8eb59fc833bd,male,e6a60ffa-b802-55ee-906d-71c0c104fb3a,2025-10-16T15:41:03.970912-05:00
4,TCGA-02-0055,da5f6940-e8ee-4fd1-a8da-4cd68e02e59c,female,a2315d09-058c-58f2-8702-daaa38542bf2,2025-10-16T15:41:03.970912-05:00


In [66]:
# =======================================================
# Extend final confounder-covariate table
# =======================================================

extended_confounder_covariates = (
    final_confounder_covariates
    .merge(
        gdc_case_demographics[
            [
                "case_submitter_id",
                "sex_at_birth",
            ]
        ],
        on="case_submitter_id",
        how="left",
        validate="one_to_one",
        sort=False,
    )
)

extended_confounder_covariates.to_csv(
    CONFOUNDER_COVARIATES_OUTPUT_PATH,
    index=False,
)

print(
    "Extended confounder-covariate table: "
    f"{extended_confounder_covariates.shape}"
)
print(
    "Saved: "
    f"{project_relative_path(CONFOUNDER_COVARIATES_OUTPUT_PATH)}"
)

Extended confounder-covariate table: (9965, 18)
Saved: data/interim/metadata/tcga_primary_tumor_multiomic_confounder_covariates.csv


In [67]:
# =======================================================
# Validate extended confounder-covariate artifact
# =======================================================

written_confounder_covariates = pd.read_csv(
    CONFOUNDER_COVARIATES_OUTPUT_PATH
)

written_sex_counts = (
    written_confounder_covariates[
        "sex_at_birth"
    ]
    .value_counts(dropna=False)
)

extension_checks = {
    "output_file_exists": (
        CONFOUNDER_COVARIATES_OUTPUT_PATH.exists()
    ),
    "shape_is_9965_by_18": (
        written_confounder_covariates.shape
        == (9965, 18)
    ),
    "case_ids_are_unique": (
        written_confounder_covariates[
            "case_submitter_id"
        ].is_unique
    ),
    "sample_order_is_preserved": (
        written_confounder_covariates[
            "final_sample_column_index"
        ].equals(
            final_confounder_covariates[
                "final_sample_column_index"
            ]
        )
    ),
    "case_order_is_preserved": (
        written_confounder_covariates[
            "case_submitter_id"
        ].equals(
            final_confounder_covariates[
                "case_submitter_id"
            ]
        )
    ),
    "sex_at_birth_is_present": (
        "sex_at_birth"
        in written_confounder_covariates.columns
    ),
    "sex_values_are_expected": (
        written_confounder_covariates[
            "sex_at_birth"
        ]
        .dropna()
        .isin(
            [
                "female",
                "male",
                "unknown",
            ]
        )
        .all()
    ),
    "raw_source_file_exists": (
        GDC_DEMOGRAPHIC_RAW_PATH.exists()
    ),
    "derived_demographic_table_exists": (
        GDC_DEMOGRAPHIC_TABLE_PATH.exists()
    ),
}

print("Demographic-extension checks:")

for check_name, check_value in extension_checks.items():
    print(f"{check_name}: {check_value}")

print(
    "\nBinary sex-at-birth availability: "
    f"{written_confounder_covariates['sex_at_birth'].isin(['female', 'male']).sum():,}"
)

print(
    "Non-binary or unavailable records: "
    f"{(~written_confounder_covariates['sex_at_birth'].isin(['female', 'male'])).sum():,}"
)

display(written_sex_counts)

Demographic-extension checks:
output_file_exists: True
shape_is_9965_by_18: True
case_ids_are_unique: True
sample_order_is_preserved: True
case_order_is_preserved: True
sex_at_birth_is_present: True
sex_values_are_expected: True
raw_source_file_exists: True
derived_demographic_table_exists: True

Binary sex-at-birth availability: 9,959
Non-binary or unavailable records: 6


sex_at_birth
female     5210
male       4749
unknown       4
NaN           2
Name: count, dtype: int64

In [68]:
# =======================================================
# Inspect confounder provenance registry entries
# =======================================================

RAW_DATA_REGISTRY_PATH = (
    Paths.config
    / "raw_data_registry.json"
)

with RAW_DATA_REGISTRY_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    raw_data_registry = json.load(file)


def find_registry_records(node, path=()):
    records = []

    if isinstance(node, dict):
        scalar_text = " ".join(
            [
                str(key)
                for key in node
            ]
            + [
                str(value)
                for value in node.values()
                if isinstance(
                    value,
                    (str, int, float, bool),
                )
            ]
        ).lower()

        if any(
            term in scalar_text
            for term in [
                "confounder",
                "panimmune",
                "absolute",
                "consensus purity",
                "leukocyte",
            ]
        ):
            records.append(
                {
                    "registry_path": ".".join(path),
                    "record": node,
                }
            )

        for key, value in node.items():
            records.extend(
                find_registry_records(
                    value,
                    path + (str(key),),
                )
            )

    elif isinstance(node, list):
        for index, value in enumerate(node):
            records.extend(
                find_registry_records(
                    value,
                    path + (str(index),),
                )
            )

    return records


confounder_registry_records = (
    find_registry_records(
        raw_data_registry
    )
)

print(
    "Registry path: "
    f"{project_relative_path(RAW_DATA_REGISTRY_PATH)}"
)
print(
    "Top-level keys: "
    f"{list(raw_data_registry)}"
)
print(
    "Matching records: "
    f"{len(confounder_registry_records)}"
)

for registry_record in confounder_registry_records:
    print(
        "\nRegistry location: "
        f"{registry_record['registry_path']}"
    )
    print(
        json.dumps(
            registry_record["record"],
            indent=2,
        )[:3000]
    )

Registry path: config/raw_data_registry.json
Top-level keys: ['depmap', 'gdsc', 'ctrp', 'prism', 'tcga', 'lincs', 'cell_model_passports', 'chembl', 'drugbank']
Matching records: 11

Registry location: tcga.external_resources
{
  "estimate": {
    "source_database": "Nature Communications supplementary data",
    "provider": "Springer Nature",
    "publication_title": "Inferring tumour purity and stromal and immune cell admixture from expression data",
    "publication_year": 2013,
    "doi": "10.1038/ncomms3612",
    "download_page_url": "https://www.nature.com/articles/ncomms3612",
    "retrieval_method": "Manual download from the publication supplementary-information section",
    "canonical_dir": "data/raw/tcga/confounders",
    "files": {
      "41467_2013_BFncomms3612_MOESM488_ESM.xlsx": {
        "supplement": "Supplementary Data 1",
        "role": "supporting_signature_definition",
        "file_url": "https://media.springernature.com/original/springer-static/esm/art%3A10.1038%

In [69]:
# =======================================================
# Register GDC case-demographic source
# =======================================================

with GDC_DEMOGRAPHIC_RAW_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    demographic_source_snapshot = json.load(file)

gdc_case_demographic_registry_entry = {
    "source_database": (
        "NCI Genomic Data Commons API"
    ),
    "provider": (
        "National Cancer Institute"
    ),
    "endpoint_url": GDC_CASES_ENDPOINT,
    "data_release": (
        demographic_source_snapshot[
            "gdc_status"
        ]["data_release"]
    ),
    "retrieval_method": (
        "Programmatic query to the GDC cases endpoint "
        "for the frozen 9,965-case TCGA cohort"
    ),
    "canonical_dir": (
        "data/raw/tcga/confounders"
    ),
    "query_scope": {
        "entity": "cases",
        "identifier": "submitter_id",
        "requested_field": (
            "demographic.sex_at_birth"
        ),
        "requested_case_count": 9965,
    },
    "files": {
        GDC_DEMOGRAPHIC_RAW_PATH.name: {
            "role": (
                "case_level_demographic_confounder_input"
            ),
            "retrieved_at_utc": (
                demographic_source_snapshot[
                    "retrieved_at_utc"
                ]
            ),
            "description": (
                "GDC case-level demographic metadata "
                "retrieved for the frozen TCGA "
                "multiomic cohort, retaining "
                "sex_at_birth for downstream "
                "sex-chromosome confounding assessment"
            ),
        }
    },
}

raw_data_registry[
    "tcga"
][
    "external_resources"
][
    "gdc_case_demographics"
] = gdc_case_demographic_registry_entry

with RAW_DATA_REGISTRY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        raw_data_registry,
        file,
        indent=2,
    )
    file.write("\n")

with RAW_DATA_REGISTRY_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    written_raw_data_registry = json.load(file)

written_demographic_registry_entry = (
    written_raw_data_registry[
        "tcga"
    ][
        "external_resources"
    ][
        "gdc_case_demographics"
    ]
)

print(
    "Registry updated: "
    f"{project_relative_path(RAW_DATA_REGISTRY_PATH)}"
)

print(
    json.dumps(
        written_demographic_registry_entry,
        indent=2,
    )
)

Registry updated: config/raw_data_registry.json
{
  "source_database": "NCI Genomic Data Commons API",
  "provider": "National Cancer Institute",
  "endpoint_url": "https://api.gdc.cancer.gov/cases",
  "data_release": "Data Release 45.0 - December 04, 2025",
  "retrieval_method": "Programmatic query to the GDC cases endpoint for the frozen 9,965-case TCGA cohort",
  "canonical_dir": "data/raw/tcga/confounders",
  "query_scope": {
    "entity": "cases",
    "identifier": "submitter_id",
    "requested_field": "demographic.sex_at_birth",
    "requested_case_count": 9965
  },
  "files": {
    "gdc_tcga_case_demographics_2026-08-06.json": {
      "role": "case_level_demographic_confounder_input",
      "retrieved_at_utc": "2026-08-06T12:20:09.023038+00:00",
      "description": "GDC case-level demographic metadata retrieved for the frozen TCGA multiomic cohort, retaining sex_at_birth for downstream sex-chromosome confounding assessment"
    }
  }
}


## Conclusion

Notebook 204 characterized the main biological and technical confounders
associated with the frozen 9,965-case TCGA multi-omic cohort.

### Main findings

- Tumor project and methylation platform define substantial cohort structure.
- Eleven projects contain both HM27 and HM450 samples, with evidence that
  platform and tumor composition can be associated within some projects.
- ABSOLUTE purity is available for 9,521 cases (95.54%) and was retained as
  the primary candidate purity covariate.
- Consensus Purity Estimate is available for 8,060 cases (80.88%) and was
  retained as an alternative sensitivity variable where project coverage is
  adequate.
- At least one external purity estimate is available for 9,800 cases (98.34%).
- PanImmune leukocyte fraction is available for 9,631 cases (96.65%), with
  structurally absent coverage in DLBC, LAML, and THYM and partial coverage
  in TGCT.
- ESTIMATE RNASeqV2 scores cover only 2,399 cases (24.07%) and are strongly
  concentrated in a subset of projects. They were therefore not incorporated
  into the final covariate table.
- The external PanImmune proliferation score is available for 9,006 cases
  (90.38%). It was retained as a reference score rather than as the definitive
  proliferation covariate.
- RNA-seq and methylation plate variables show project-dependent technical
  structure and were retained for lineage-aware robustness analyses.
- RNA center is largely nested within project, methylation center is constant,
  and tissue source site is too granular for routine fixed-effect adjustment.

Purity, leukocyte infiltration, proliferation, platform, and technical
variables showed project-dependent behavior. They should therefore be
evaluated within lineage rather than through naïve pan-cancer adjustment.

### Demographic-sex extension

A downstream program-discovery assessment identified candidate methylation
components with substantial chrX loading enrichment. To support an explicit
sensitivity analysis, case-level `sex_at_birth` was retrieved from the NCI
Genomic Data Commons cases API for the complete frozen cohort.

The GDC query returned one record for each of the 9,965 frozen cases:

- 5,210 cases recorded as `female`;
- 4,749 cases recorded as `male`;
- 4 cases recorded as `unknown`;
- 2 cases without an available value.

Binary `sex_at_birth` information is therefore available for 9,959 cases.
The six unknown or unavailable records were retained without imputation.

`sex_at_birth` is case-level demographic metadata. It supports sensitivity
assessment of sex-associated molecular structure but should not be interpreted
as a direct measurement of tumor sex-chromosome composition or karyotype.

No samples were excluded, no missing covariates were imputed, and no
residualization or batch correction was performed in this notebook.

### Published artifacts

The raw GDC demographic response was frozen at:

`data/raw/tcga/confounders/gdc_tcga_case_demographics_2026-08-06.json`

The normalized case-level demographic table was written to:

`data/interim/metadata/tcga_primary_tumor_case_demographics.csv`

The downstream confounder-covariate table was updated at:

`data/interim/metadata/tcga_primary_tumor_multiomic_confounder_covariates.csv`

The final table contains 9,965 cases and 18 variables, preserving biological
covariates, alternative sensitivity estimates, demographic sex-at-birth, and
technical descriptors required for program discovery and robustness analyses
in notebooks 205 and 206.

The demographic source and GDC Data Release 45.0 provenance were registered
in:

`config/raw_data_registry.json`